# 스트레스 점수 예측 V16 — OOF 검증 포함

**V14 Blend95 98% + V15 2% Micro Blend**

- V14, V15, V16을 동일한 두 가지 5-Fold 분할에서 직접 비교
- V14 OOF: 최고 모델 95% + V10 계열 보조 모델 5%
- V15 OOF: 3개 시드 ExtraTrees의 50% 분위수 중앙값
- V16 OOF: V14 98% + V15 2%
- 모든 전처리는 각 Fold의 Train 데이터에서만 학습
- 최종 제출 파일은 기존 V14·V15 제출 예측을 98:2로 결합

> ExtraTrees를 반복 학습하므로 실행 시간이 오래 걸릴 수 있습니다.

In [6]:
from pathlib import Path
import os

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

## 1. 설정

In [7]:
DATA_DIR = Path("/Users/bitsaem/Desktop/stress_project/open")
OUTPUT_DIR = Path("/Users/bitsaem/Desktop/stress_project")

TARGET = "stress_score"
ID_COLUMN = "ID"
MODEL_SEEDS = (42, 77, 2026)
CV_SEEDS = (42, 2026)
N_ESTIMATORS = 1200

BASE_QUANTILE = 0.510
POOLED_QUANTILE = 0.505
AGE_QUANTILE = 0.500
V15_QUANTILE = 0.500

V14_BASE_WEIGHT = 0.95
V14_AUX_WEIGHT = 0.05
V16_V14_WEIGHT = 0.98
V16_V15_WEIGHT = 0.02

WEIGHTED_FEATURES = [
    "mean_working", "bmi", "cholesterol", "height",
    "glucose", "weight", "cholesterol_glucose_ratio",
    "bone_density",
]
AGE_WEIGHTED_FEATURES = WEIGHTED_FEATURES + [
    "mean_arterial_pressure", "diastolic_blood_pressure",
]

## 2. 데이터와 파생변수

In [8]:
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
submission = pd.read_csv(DATA_DIR / "sample_submission.csv")

if set(train.columns) - {TARGET} != set(test.columns):
    raise ValueError("Train과 Test 입력 컬럼이 다릅니다.")
if not submission[ID_COLUMN].equals(test[ID_COLUMN]):
    raise ValueError("제출 양식과 Test의 ID 순서가 다릅니다.")


def add_base_features(frame):
    result = frame.copy()
    height_m = result["height"] / 100.0
    sbp = result["systolic_blood_pressure"]
    dbp = result["diastolic_blood_pressure"]
    result["bmi"] = result["weight"] / height_m.pow(2)
    result["pulse_pressure"] = sbp - dbp
    result["mean_arterial_pressure"] = (sbp + 2.0 * dbp) / 3.0
    result["blood_pressure_ratio"] = sbp / (dbp + 1e-6)
    result["cholesterol_glucose_ratio"] = (
        result["cholesterol"] / (result["glucose"] + 1e-6)
    )
    result["missing_count"] = result.isna().sum(axis=1)
    return result


def add_weighted_features(frame):
    result = add_base_features(frame)
    for feature in WEIGHTED_FEATURES:
        result[f"{feature}__copy1"] = result[feature]
        result[f"{feature}__copy2"] = result[feature]
    return result


def add_age_features(frame):
    result = add_base_features(frame)
    result["age_bmi"] = result["age"] * result["bmi"]
    result["age_glucose"] = result["age"] * result["glucose"]
    result["age_cholesterol"] = result["age"] * result["cholesterol"]
    result["age_bone_density"] = result["age"] * result["bone_density"]
    result["age_mean_arterial_pressure"] = (
        result["age"] * result["mean_arterial_pressure"]
    )
    for feature in AGE_WEIGHTED_FEATURES:
        result[f"{feature}__copy1"] = result[feature]
    return result


y = train[TARGET].reset_index(drop=True)
raw_train = train.drop(columns=[TARGET, ID_COLUMN]).reset_index(drop=True)
X_weighted = add_weighted_features(raw_train)
X_age = add_age_features(raw_train)
print("Weighted:", X_weighted.shape, "| Age:", X_age.shape)

Weighted: (3000, 38) | Age: (3000, 37)


## 3. Train 전용 전처리

In [9]:
def make_preprocessor(frame):
    categorical = frame.select_dtypes(
        include=["object", "category", "string"]
    ).columns.tolist()
    numerical = frame.select_dtypes(include=["number"]).columns.tolist()
    return ColumnTransformer(
        [
            (
                "categorical",
                Pipeline(
                    [
                        (
                            "imputer",
                            SimpleImputer(
                                strategy="constant",
                                fill_value="missing",
                            ),
                        ),
                        (
                            "encoder",
                            OneHotEncoder(
                                handle_unknown="ignore",
                                sparse_output=False,
                            ),
                        ),
                    ]
                ),
                categorical,
            ),
            (
                "numerical",
                SimpleImputer(strategy="median", add_indicator=True),
                numerical,
            ),
        ]
    )


def tree_matrix(model, matrix):
    return np.column_stack(
        [tree.predict(matrix) for tree in model.estimators_]
    )


def q(matrix, quantile):
    return np.clip(
        np.quantile(matrix, quantile, axis=1),
        0.0,
        1.0,
    )

## 4. 동일 Fold에서 V14·V15·V16 OOF 검증

In [10]:
score_rows = []
oof_by_split = {}

for cv_seed in CV_SEEDS:
    cv = KFold(n_splits=5, shuffle=True, random_state=cv_seed)
    oof_v14 = np.zeros(len(y))
    oof_v15 = np.zeros(len(y))
    oof_v16 = np.zeros(len(y))
    fold_scores = []

    for fold, (train_idx, valid_idx) in enumerate(cv.split(X_weighted), start=1):
        weighted_preprocessor = make_preprocessor(
            X_weighted.iloc[train_idx]
        )
        weighted_train = weighted_preprocessor.fit_transform(
            X_weighted.iloc[train_idx]
        )
        weighted_valid = weighted_preprocessor.transform(
            X_weighted.iloc[valid_idx]
        )

        matrices = {}
        for model_seed in MODEL_SEEDS:
            model = ExtraTreesRegressor(
                n_estimators=N_ESTIMATORS,
                min_samples_leaf=1,
                max_features=1,
                random_state=model_seed,
                n_jobs=-1,
            )
            model.fit(weighted_train, y.iloc[train_idx])
            matrices[model_seed] = tree_matrix(
                model,
                weighted_valid,
            )

        pred_base = q(matrices[42], BASE_QUANTILE)
        pred_pooled = q(
            np.concatenate(
                [matrices[seed] for seed in MODEL_SEEDS],
                axis=1,
            ),
            POOLED_QUANTILE,
        )
        pred_v15_fold = np.median(
            np.column_stack(
                [
                    q(matrices[seed], V15_QUANTILE)
                    for seed in MODEL_SEEDS
                ]
            ),
            axis=1,
        )

        age_preprocessor = make_preprocessor(X_age.iloc[train_idx])
        age_train = age_preprocessor.fit_transform(
            X_age.iloc[train_idx]
        )
        age_valid = age_preprocessor.transform(
            X_age.iloc[valid_idx]
        )
        age_model = ExtraTreesRegressor(
            n_estimators=N_ESTIMATORS,
            min_samples_leaf=1,
            max_features=1,
            random_state=42,
            n_jobs=-1,
        )
        age_model.fit(age_train, y.iloc[train_idx])
        pred_age = q(
            tree_matrix(age_model, age_valid),
            AGE_QUANTILE,
        )

        pred_aux = np.clip(
            0.70 * pred_pooled + 0.30 * pred_age,
            0.0,
            1.0,
        )
        pred_v14_fold = np.clip(
            V14_BASE_WEIGHT * pred_base
            + V14_AUX_WEIGHT * pred_aux,
            0.0,
            1.0,
        )
        pred_v16_fold = np.clip(
            V16_V14_WEIGHT * pred_v14_fold
            + V16_V15_WEIGHT * pred_v15_fold,
            0.0,
            1.0,
        )

        oof_v14[valid_idx] = pred_v14_fold
        oof_v15[valid_idx] = pred_v15_fold
        oof_v16[valid_idx] = pred_v16_fold
        fold_scores.append(
            {
                "CV_seed": cv_seed,
                "Fold": fold,
                "V14_MAE": mean_absolute_error(
                    y.iloc[valid_idx], pred_v14_fold
                ),
                "V15_MAE": mean_absolute_error(
                    y.iloc[valid_idx], pred_v15_fold
                ),
                "V16_MAE": mean_absolute_error(
                    y.iloc[valid_idx], pred_v16_fold
                ),
            }
        )
        print(f"CV seed {cv_seed} | Fold {fold}: complete")

    v14_mae = mean_absolute_error(y, oof_v14)
    v15_mae = mean_absolute_error(y, oof_v15)
    v16_mae = mean_absolute_error(y, oof_v16)
    score_rows.append(
        {
            "CV_seed": cv_seed,
            "V14_MAE": v14_mae,
            "V15_MAE": v15_mae,
            "V16_MAE": v16_mae,
            "V16_vs_V14": v16_mae - v14_mae,
        }
    )
    oof_by_split[cv_seed] = {
        "v14": oof_v14,
        "v15": oof_v15,
        "v16": oof_v16,
        "fold_scores": pd.DataFrame(fold_scores),
    }

score_table = pd.DataFrame(score_rows)
print("\n반복 CV 결과")
display(score_table)
print("V14 평균 MAE:", f"{score_table['V14_MAE'].mean():.6f}")
print("V15 평균 MAE:", f"{score_table['V15_MAE'].mean():.6f}")
print("V16 평균 MAE:", f"{score_table['V16_MAE'].mean():.6f}")
print(
    "V16 vs V14 평균:",
    f"{score_table['V16_vs_V14'].mean():+.6f}",
)
print(
    "V16이 V14를 이긴 CV 분할:",
    int((score_table["V16_vs_V14"] < 0).sum()),
    "/",
    len(score_table),
)

CV seed 42 | Fold 1: complete
CV seed 42 | Fold 2: complete
CV seed 42 | Fold 3: complete
CV seed 42 | Fold 4: complete
CV seed 42 | Fold 5: complete
CV seed 2026 | Fold 1: complete
CV seed 2026 | Fold 2: complete
CV seed 2026 | Fold 3: complete
CV seed 2026 | Fold 4: complete
CV seed 2026 | Fold 5: complete

반복 CV 결과


,CV_seed,V14_MAE,V15_MAE,V16_MAE,V16_vs_V14
0,42,0.149080,0.148860,0.149072,-0.000008
1,2026,0.145586,0.145343,0.145578,-0.000008


V14 평균 MAE: 0.147333
V15 평균 MAE: 0.147102
V16 평균 MAE: 0.147325
V16 vs V14 평균: -0.000008
V16이 V14를 이긴 CV 분할: 2 / 2


## 5. 기존 제출 예측 98:2 결합

In [11]:
V14_PATH = OUTPUT_DIR / "submit_v14_blend95.csv"
V15_PATH = OUTPUT_DIR / "submit_v15_repeated_cv_multiseed.csv"
OUTPUT_PATH = OUTPUT_DIR / "submit_v16_v14_98_v15_02.csv"

v14_submission = pd.read_csv(V14_PATH)
v15_submission = pd.read_csv(V15_PATH)
if not v14_submission[ID_COLUMN].equals(v15_submission[ID_COLUMN]):
    raise ValueError("V14와 V15의 ID 또는 순서가 다릅니다.")

pred_v16 = np.clip(
    V16_V14_WEIGHT * v14_submission[TARGET].to_numpy()
    + V16_V15_WEIGHT * v15_submission[TARGET].to_numpy(),
    0.0,
    1.0,
)
submission_v16 = v14_submission.copy()
submission_v16[TARGET] = pred_v16
submission_v16.to_csv(OUTPUT_PATH, index=False, encoding="utf-8")

movement = pred_v16 - v14_submission[TARGET].to_numpy()
print("Saved:", OUTPUT_PATH)
print("V14 대비 평균 절대 변화:", f"{np.abs(movement).mean():.8f}")
print("첫 10개 예측값:", pred_v16[:10])
submission_v16.head(10)

Saved: /Users/bitsaem/Desktop/stress_project/submit_v16_v14_98_v15_02.csv
V14 대비 평균 절대 변화: 0.00012533
첫 10개 예측값: [0.560939 0.97     0.19     0.440147 0.53     0.4599   0.78     0.469706
 0.54969  0.88    ]


,ID,stress_score
0,TEST_0000,0.560939
1,TEST_0001,0.970000
2,TEST_0002,0.190000
3,TEST_0003,0.440147
4,TEST_0004,0.530000
5,TEST_0005,0.459900
6,TEST_0006,0.780000
7,TEST_0007,0.469706
8,TEST_0008,0.549690
9,TEST_0009,0.880000


## 제출 기준

- `V16이 V14를 이긴 CV 분할: 2 / 2`일 때만 V16을 제출 후보로 고려
- `V16 vs V14 평균`이 음수일수록 로컬 MAE가 개선된 것
- 조건을 만족하지 않으면 현재 1등 V14 Blend95를 유지